# 2.3 · PD results

**How the trained Experiment 2 arms score on the held-out real PD datasets, and the released frontier reference.** Reads the phase-2 benchmark under `output/results/pd/eval/`; it is **safe to run before the benchmark exists** — every figure shows a labelled placeholder until then, because phase 2 cannot start until phase 1 has trained every arm.

**Why the reference sits in the same table.** The released TabICLv2, TabPFN-3, CatBoost and a linear model are scored by the *same* code, same day, same context cap and splits as our arms — every comparison this project got wrong, it got wrong by scoring the two sides through different paths. Our nano-scale arms are expected to trail the frontier in absolute terms; the claim is about **prior contrast at matched compute**, not absolute SOTA.

**How to read the metrics.** PD is imbalanced, so accuracy is never reported alone — at a 7% base rate 'never defaults' already scores 0.93, and Tanna 2026 shows default-threshold GBDTs collapse to 0% recall (`papers/2026/05_Tanna_DataPresentation`). ROC-AUC ranks, and ECE/Brier/calibration-slope check the probabilities a PD model actually needs.

*Grounding.* Cited by path against `tfm-library` pin `52dab01`; the credit-domain models behind our prior (Merton/Vasicek, Basel) are **external** to the library and marked so.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.visualize import results_plots, figures, style, literature, literature_plots

style.apply()   # ONE shared style: identical colours in every figure of every notebook
pd.set_option("display.width", 200, "display.max_columns", 40)

TASK = "pd"
EXP = "exp2"
# Every started arm leaves output/manifests/exp2_pd__<run>__progress.csv and __telemetry.csv;
# these read them, so a PARTIAL sweep still plots. Clears THIS notebook's figure folder first.
FIGS = figures.FigureSaver("2.3_pd_results")

## 0. The colour key

In [ ]:
FIGS.save(style.show_palette(), "palette",
    caption="The shared colour vocabulary used on every axis of this notebook: each swatch names the prior, data source, literature overlay or annotation it marks.");

## 1. Overall ranking

Every model on one axis, sorted, coloured by kind — our credit arms, the control, and the external baselines — with the seed spread as an error bar so nothing is ranked inside noise.

In [ ]:
FIGS.save(results_plots.overall_ranking(TASK, exp=EXP), "overall_ranking",
    caption="Mean ROC-AUC per configuration on development datasets, averaged over evaluation and training seeds. Error bars show the standard deviation across training seeds.");

## 2. Do we clear the frontier?

Our nano-scale arms against the released TabICLv2 specifically: the bar is each arm's score minus the reference, so anything right of zero beats a frontier model trained on orders of magnitude more compute. Trailing it is expected and not the claim.

In [ ]:
FIGS.save(results_plots.beats_reference(TASK, exp=EXP), "beats_reference",
    caption="Each trained arm's mean AUC minus the released TabICLv2's, one horizontal bar per arm, with a reference line at zero marking the frontier model.");

## 3. Every metric by model kind

The whole scoreboard at once: one panel per benchmark metric, a bar per model kind. A prior that helps ranking but hurts calibration shows up as a split across panels — the calibration axis the library says is under-measured (`SYNTHESIS.md`).

In [ ]:
FIGS.save(results_plots.metric_grid(TASK, exp=EXP), "metric_grid",
    caption="One panel per benchmark metric, each a bar per model kind (credit, control, baseline); the arrow in each panel title marks the improving direction.");

## 4. Per-dataset breakdown

No dataset hidden behind a mean: the best of each kind on each real dataset, side by side, so a prior that wins on average by helping one easy dataset is exposed.

In [ ]:
FIGS.save(results_plots.per_dataset(TASK, exp=EXP), "per_dataset",
    caption="Best AUC per model kind on each real dataset, as grouped bars, one group per dataset.");

## 5. Per-dataset heatmap

The same per-dataset scores as a heatmap, so the pattern across all datasets and kinds is one glance rather than a wall of bars.

In [ ]:
FIGS.save(results_plots.per_dataset_heatmap(TASK, exp=EXP), "per_dataset_heatmap",
    caption="Best AUC of each model kind on each dataset as an annotated heatmap, datasets on the vertical axis and kinds on the horizontal.");

## 6. Credit prior versus control

The figure the whole Experiment answers: the distribution of per-model scores for credit arms, control arms and baselines, each point a model and the bar its group mean. A win here must clear the control, which is TabICLv2's own prior by construction.

In [ ]:
FIGS.save(results_plots.credit_vs_control(TASK, exp=EXP), "credit_vs_control",
    caption="Distribution of per-model mean AUC for credit-prior arms, control arms and external baselines; one point per model with a bar at each group mean.");

## 7. Effect of each fine-tuning lever

Exp2's own question: which knob moved the score. The benchmark metric grouped by credit fraction, freeze strategy, L2-SP and learning rate — the swept levers whose literature-grounded defaults are set in `docs/CONFIG_REFERENCE.md`.

In [ ]:
FIGS.save(results_plots.lever_effect(TASK, exp=EXP), "lever_effect",
    caption="Mean AUC grouped by each fine-tuning lever in turn \u2014 credit fraction, freeze strategy, L2-SP alpha and learning rate \u2014 one panel per lever.");

## 8. Where the field sits on credit data

The benchmark above is this project's own. This figure places it against the **only credit-domain numbers the `tfm-library` contains**. Tanna 2026 reports ROC-AUC on the real Home Credit and Lending Club books (`papers/2026/05_Tanna_DataPresentation` Table 3 — TabICL 0.771, TabPFN 0.786 on Home Credit; the RandomForest baseline the TFMs chase sits at 0.739, §5.3), and Hollmann 2023 reports Credit-g at 0.789 (`papers/2023/09_Hollmann_TabPFN` Table 2). **Read it as a landscape, not a scoreboard:** each value is measured under that paper's own full-dataset protocol — tens of thousands of context rows — not the 1024-row in-context setting the arms above use, so it grounds the *regime*, not a like-for-like target. The bars are drawn in `literature.py`'s REFERENCE teal for an evaluated result and EXTERNAL amber for a described ceiling, the same provenance vocabulary the colour key names.

In [ ]:
FIGS.save(literature_plots.credit_benchmark_landscape(), "literature_landscape",
    caption="Reported ROC-AUC on real credit datasets drawn from the tfm-library, one horizontal bar per published result, each on its paper's own full-dataset protocol rather than the 1024-row in-context setting used elsewhere in this notebook.");

## Summary

A printed recap of the benchmark, the tfm-library sources this notebook leans on, and the figure inventory.

In [ ]:
print(results_plots.results_summary(TASK, exp=EXP))
print()
print(literature.references_md(["oprior_ticlv2", "tanna_resampling", "tanna_paradox", "calibration_gap", "purucker_highcard", "merton_vasicek"]))
print()
print(literature_plots.summary())
print()
print(FIGS.summary())